# Fake News Classifier
### LR | NB | RF | XGB × TF-IDF | BOW | Lemmatization

**Workflow:**
1. Imports
2. Data inladen
3. Train / Validatie / Test split
4. Preprocessing
5. Feature Extraction (TF-IDF en BOW)
6. Modellen trainen (LR, NB, RF, XGB)
7. Evaluatie: Accuracy, F1, Cross-validatie
8. Visualisatie
9. Voorspellingen op nieuwe data (Kaggle submission)

## STAP 1 - Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import pickle

import nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

print('Setup ready!')

## STAP 2 - Data inladen en bekijken

In [ ]:
df = pd.read_csv('/Users/domiendarmont/Desktop/Ironhack/lab_week4_NLP/TEST:W4_project_NLP/training_data_fixed.csv')
df.fillna('', inplace=True)

print('Shape:', df.shape)
print('Lege waarden:', df.isnull().sum().to_dict())
print('Klasse verdeling:')
print(df['label'].value_counts())
df.head()

## STAP 3 - Train / Validatie / Test Split
**Belangrijk:** split VOOR preprocessing zodat test en validatie echt ongezien blijven!

```
Alle data (100%)
├── Train      (64%) → model leert
├── Validatie  (16%) → model tunen en vergelijken
└── Test       (20%) → finale evaluatie
```

In [ ]:
# Stap 1: splits in train (80%) en test (20%)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

# Stap 2: splits train verder in train (80%) en validatie (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

print(f'Train:      {X_train.shape[0]} samples ({X_train.shape[0]/len(df)*100:.0f}%)')
print(f'Validatie:  {X_val.shape[0]} samples ({X_val.shape[0]/len(df)*100:.0f}%)')
print(f'Test:       {X_test.shape[0]} samples ({X_test.shape[0]/len(df)*100:.0f}%)')

## STAP 4 - Preprocessing
Data is al lowercase, geen HTML. Stappen: special chars → tokenization → stopwords → lemmatization

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Special characters verwijderen
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Losse letters en dubbele spaties
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenization
    tokens = word_tokenize(text)
    # Punctuation removal
    tokens = [w for w in tokens if w not in string.punctuation]
    # Stopwords removal
    tokens = [w for w in tokens if w not in stop_words]
    # Lemmatization
    tokens = [lemmatizer.lemmatize(w) for w in tokens]
    return ' '.join(tokens)

# Toepassen op train, validatie en test APART
X_train = X_train.apply(preprocess)
X_val   = X_val.apply(preprocess)
X_test  = X_test.apply(preprocess)

# Visuele check voor vs na preprocessing
print('VOOR:', df['text'].loc[X_train.index[1]])
print('NA:  ', X_train.values[1])

## STAP 5 - Feature Extraction
**fit_transform op train** → leert vocabulaire  
**transform op validatie en test** → gebruikt ZELFDE vocabulaire, NOOIT fit_transform!

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)
X_test_tfidf  = tfidf.transform(X_test)

# BOW
cv = CountVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_bow = cv.fit_transform(X_train)
X_val_bow   = cv.transform(X_val)
X_test_bow  = cv.transform(X_test)

print('TF-IDF shape:', X_train_tfidf.shape)
print('BOW shape:   ', X_train_bow.shape)

## STAP 6 - Modellen Trainen en Evalueren
Alle modellen getraind en geëvalueerd op train, validatie en test met Accuracy en F1 score.

In [ ]:
modellen = {
    'LR + TF-IDF':  (LogisticRegression(max_iter=1000),                                      X_train_tfidf, X_val_tfidf, X_test_tfidf),
    'NB + TF-IDF':  (MultinomialNB(),                                                         X_train_tfidf, X_val_tfidf, X_test_tfidf),
    'LR + BOW':     (LogisticRegression(max_iter=1000),                                      X_train_bow,   X_val_bow,   X_test_bow),
    'NB + BOW':     (MultinomialNB(),                                                         X_train_bow,   X_val_bow,   X_test_bow),
    'RF + TF-IDF':  (RandomForestClassifier(n_estimators=100, random_state=42),              X_train_tfidf, X_val_tfidf, X_test_tfidf),
    'RF + BOW':     (RandomForestClassifier(n_estimators=100, random_state=42),              X_train_bow,   X_val_bow,   X_test_bow),
    'XGB + TF-IDF': (XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'), X_train_tfidf, X_val_tfidf, X_test_tfidf),
    'XGB + BOW':    (XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss'), X_train_bow,   X_val_bow,   X_test_bow),
}

resultaten = {}
getrainde_modellen = {}

print(f"{'Model':<15} {'Tr Acc':>8} {'Val Acc':>8} {'Te Acc':>8} {'Tr F1':>8} {'Val F1':>8} {'Te F1':>8} {'Overfit':>8}")
print('-' * 80)

for naam, (model, X_tr, X_v, X_te) in modellen.items():
    model.fit(X_tr, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_tr))
    val_acc   = accuracy_score(y_val,   model.predict(X_v))
    test_acc  = accuracy_score(y_test,  model.predict(X_te))
    train_f1  = f1_score(y_train,       model.predict(X_tr))
    val_f1    = f1_score(y_val,         model.predict(X_v))
    test_f1   = f1_score(y_test,        model.predict(X_te))
    overfit   = abs(train_acc - test_acc)

    resultaten[naam] = (train_acc, val_acc, test_acc, train_f1, val_f1, test_f1)
    getrainde_modellen[naam] = model

    print(f"{naam:<15} {train_acc:>8.3f} {val_acc:>8.3f} {test_acc:>8.3f} {train_f1:>8.3f} {val_f1:>8.3f} {test_f1:>8.3f} {overfit:>8.3f}")

## STAP 7 - Cross Validatie
Bevestigt dat de scores betrouwbaar zijn en niet afhangen van één specifieke split.

In [ ]:
modellen_cv = {
    'LR + TF-IDF': (LogisticRegression(max_iter=1000), X_train_tfidf),
    'NB + TF-IDF': (MultinomialNB(),                   X_train_tfidf),
    'LR + BOW':    (LogisticRegression(max_iter=1000), X_train_bow),
    'NB + BOW':    (MultinomialNB(),                   X_train_bow),
}

print(f"{'Model':<15} {'CV Gemiddelde':>15} {'Std':>8} {'Scores'}")
print('-' * 65)

for naam, (model, X_tr) in modellen_cv.items():
    scores = cross_val_score(model, X_tr, y_train, cv=5, scoring='accuracy')
    print(f"{naam:<15} {scores.mean():>15.3f} {scores.std():>8.3f}   {scores.round(3)}")

## STAP 8 - Visualisatie

In [ ]:
namen      = list(resultaten.keys())
n_modellen = len(namen)
x          = np.arange(n_modellen)
breedte    = 0.25

train_accs = [v[0] for v in resultaten.values()]
val_accs   = [v[1] for v in resultaten.values()]
test_accs  = [v[2] for v in resultaten.values()]
train_f1s  = [v[3] for v in resultaten.values()]
val_f1s    = [v[4] for v in resultaten.values()]
test_f1s   = [v[5] for v in resultaten.values()]

fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Accuracy per model - train vs val vs test
axes[0].bar(x - breedte, train_accs, breedte, label='Train',      color='steelblue')
axes[0].bar(x,           val_accs,   breedte, label='Validatie',  color='orange')
axes[0].bar(x + breedte, test_accs,  breedte, label='Test',       color='coral')
axes[0].set_title('Accuracy per model: Train vs Validatie vs Test', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(namen, rotation=15)
axes[0].set_ylim(0.85, 1.01)
axes[0].set_ylabel('Accuracy')
axes[0].axhline(y=0.9, color='green', linestyle='--', label='90% threshold')
for i, (tr, va, te) in enumerate(zip(train_accs, val_accs, test_accs)):
    axes[0].text(i - breedte, tr + 0.002, f'{tr:.3f}', ha='center', fontsize=7)
    axes[0].text(i,           va + 0.002, f'{va:.3f}', ha='center', fontsize=7)
    axes[0].text(i + breedte, te + 0.002, f'{te:.3f}', ha='center', fontsize=7)
axes[0].legend()

# F1 per model - train vs val vs test
axes[1].bar(x - breedte, train_f1s, breedte, label='Train',      color='steelblue')
axes[1].bar(x,           val_f1s,   breedte, label='Validatie',  color='orange')
axes[1].bar(x + breedte, test_f1s,  breedte, label='Test',       color='coral')
axes[1].set_title('F1 Score per model: Train vs Validatie vs Test', fontsize=13, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(namen, rotation=15)
axes[1].set_ylim(0.85, 1.01)
axes[1].set_ylabel('F1 Score')
axes[1].axhline(y=0.9, color='green', linestyle='--', label='90% threshold')
for i, (tr, va, te) in enumerate(zip(train_f1s, val_f1s, test_f1s)):
    axes[1].text(i - breedte, tr + 0.002, f'{tr:.3f}', ha='center', fontsize=7)
    axes[1].text(i,           va + 0.002, f'{va:.3f}', ha='center', fontsize=7)
    axes[1].text(i + breedte, te + 0.002, f'{te:.3f}', ha='center', fontsize=7)
axes[1].legend()

plt.suptitle('Overzicht Alle Modellen', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Classification report en confusion matrix beste model (LR + BOW)
best_model  = getrainde_modellen['LR + BOW']
y_pred_best = best_model.predict(X_test_bow)

print('=== Classification Report: LR + BOW ===')
print(classification_report(y_test, y_pred_best, target_names=['Fake (0)', 'Real (1)']))

plt.figure(figsize=(6, 4))
sns.heatmap(confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.title('Confusion Matrix - LR + BOW (beste model)')
plt.xlabel('Voorspeld')
plt.ylabel('Werkelijk')
plt.show()

## STAP 9 - Voorspellingen op nieuwe testdata (Kaggle submission)

In [ ]:
# Laad nieuwe testdata
df_new = pd.read_csv('/Users/domiendarmont/Desktop/Ironhack/lab_week4_NLP/TEST:W4_project_NLP/testing_data_lowercase_nolabels.csv',
                     header=None, sep='\t', names=['id', 'text'])

# Preprocessing - zelfde functie als traindata!
df_new['text'] = df_new['text'].apply(preprocess)

# Transformeer met ZELFDE vectorizer - NOOIT fit_transform!
X_new = cv.transform(df_new['text'])

# Voorspel met beste model
df_new['label'] = best_model.predict(X_new)

print('Voorspelling verdeling:')
print(f'Fake: {(df_new["label"]==0).sum()} ({(df_new["label"]==0).mean()*100:.1f}%)')
print(f'Real: {(df_new["label"]==1).sum()} ({(df_new["label"]==1).mean()*100:.1f}%)')

# Opslaan voor submission
df_new[['id', 'label']].to_csv('submission.csv', index=False)
print('\nOpgeslagen als submission.csv')
df_new.head()

## STAP 10 - Model opslaan

In [ ]:
with open('model_lr_bow.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('vectorizer_bow.pkl', 'wb') as f:
    pickle.dump(cv, f)

print('Model en vectorizer opgeslagen!')